# Ingresos históricos de Renca

La carga homologa todos los ejercicios al contrato canónico 2025. Los campos `source_*` conservan los valores originales y `reporte_normalizacion` permite auditar las reglas aplicadas. Las transferencias de 2002–2007 marcadas como `review` requieren cautela porque el esquema histórico no siempre distingue inequívocamente corriente y capital.

La documentación completa está en [`docs/NORMALIZACION_DATOS.md`](docs/NORMALIZACION_DATOS.md).

In [57]:
from importlib import reload

import utils.data_schema as data_schema
import utils.income_mapping as income_mapping
import utils.ip_utils as ip_utils
import utils.data as data_utils

reload(data_schema)
reload(income_mapping)
reload(ip_utils)
reload(data_utils)

# data_loader siempre homologa los Excel al contrato 2025. Como este EDA
# estudia Renca, filtramos al cargar para reducir el uso de memoria.
presupuesto = data_utils.data_loader(municipality="Renca")
reporte_normalizacion = data_utils.get_normalization_report(presupuesto)
reporte_normalizacion[[
    "source_file",
    "years",
    "review_income_rows",
    "cache_status",
]]

Normalizando 25 archivos al contrato 2025 desde /Users/m5/Documents/GitHub/proyeccion-ingresos-municipales/data
  - BD Presupuestaria 2025_09-06-2026.xlsx: 336 filas (2025; hit)
  - BD Presupuestaria 2026_2026-07-23.xlsx: 297 filas (2026; hit)
  - BD Presupuestario SM 2023_2025-01-20.xlsx: 344 filas (2023; hit)
  - BD_Presupuestaria_2002.xlsx: 165 filas (2002; hit)
  - BD_Presupuestaria_2003.xlsx: 156 filas (2003; hit)
  - BD_Presupuestaria_2004.xlsx: 165 filas (2004; hit)
  - BD_Presupuestaria_2005.xlsx: 154 filas (2005; hit)
  - BD_Presupuestaria_2006.xlsx: 161 filas (2006; hit)
  - BD_Presupuestaria_2007.xlsx: 163 filas (2007; hit)
  - BD_Presupuestaria_2008.xlsx: 210 filas (2008; hit)
  - BD_Presupuestaria_2009.xlsx: 200 filas (2009; hit)
  - BD_Presupuestaria_2010.xlsx: 294 filas (2010; hit)
  - BD_Presupuestaria_2011.xlsx: 263 filas (2011; hit)
  - BD_Presupuestaria_2012.xlsx: 245 filas (2012; hit)
  - BD_Presupuestaria_2013.xlsx: 227 filas (2013; hit)
  - BD_Presupuestaria_2014.

,source_file,years,review_income_rows,cache_status
0,BD Presupuestaria 2025_09-06-2026.xlsx,2025,0,hit
1,BD Presupuestaria 2026_2026-07-23.xlsx,2026,0,hit
2,BD Presupuestario SM 2023_2025-01-20.xlsx,2023,0,hit
3,BD_Presupuestaria_2002.xlsx,2002,1065,hit
4,BD_Presupuestaria_2003.xlsx,2003,1054,hit
5,BD_Presupuestaria_2004.xlsx,2004,1045,hit
6,BD_Presupuestaria_2005.xlsx,2005,1036,hit
7,BD_Presupuestaria_2006.xlsx,2006,1084,hit
8,BD_Presupuestaria_2007.xlsx,2007,1106,hit
9,BD_Presupuestaria_2008.xlsx,2008,0,hit


In [58]:
import utils.municipal_income as income_utils

reload(income_utils)
cobertura_anual = income_utils.infer_year_coverage(presupuesto)
cobertura_anual

,Ejercicio,last_reported_month,last_reported_month_number,year_status,is_complete
0,2002,Diciembre,12,complete,True
1,2003,Diciembre,12,complete,True
2,2004,Diciembre,12,complete,True
3,2005,Diciembre,12,complete,True
4,2006,Diciembre,12,complete,True
5,2007,Diciembre,12,complete,True
6,2008,Diciembre,12,complete,True
7,2009,Diciembre,12,complete,True
8,2010,Diciembre,12,complete,True
9,2011,Diciembre,12,complete,True


In [59]:
from importlib import reload
from IPython.display import HTML, display

import utils.plots as income_plots

reload(income_plots)

# EXCLUDED_YEARS = set()

historial_renca = income_utils.build_municipality_income_history(
    presupuesto,
    municipality="Renca",
    # complete_years_only=True,
    # exclude_years=EXCLUDED_YEARS,
)
fig_renca = income_plots.plot_annual_income_share(historial_renca)

# Conserva suficiente espacio por año y agrega desplazamiento horizontal
# cuando el gráfico es más ancho que la salida del notebook.
numero_anios = historial_renca["Ejercicio"].nunique()
ancho_grafico = max(900, numero_anios * 90 + 240)
fig_renca.update_layout(width=ancho_grafico, autosize=False)

grafico_html = fig_renca.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    config={"responsive": False, "displaylogo": False},
)
display(
    HTML(
        f'<div style="max-width: 100%; overflow-x: auto; padding-bottom: 0.5rem;">'
        f"{grafico_html}</div>"
    )
)

## Evolución porcentual por grupo de ingreso

In [60]:
reload(income_plots)

fig_renca_lineas = income_plots.plot_annual_income_share_lines(
    historial_renca
)
fig_renca_lineas.update_layout(width=ancho_grafico, autosize=False)

grafico_lineas_html = fig_renca_lineas.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    config={"responsive": False, "displaylogo": False},
)
display(
    HTML(
        f'<div style="max-width: 100%; overflow-x: auto; padding-bottom: 0.5rem;">'
        f"{grafico_lineas_html}</div>"
    )
)

## Evolución porcentual trimestral desde 2014

In [61]:
reload(income_utils)
reload(income_plots)

historial_trimestral_renca = (
    income_utils.build_municipality_quarterly_income_history(
        presupuesto,
        municipality="Renca",
        start_year=2014,
    )
)
fig_renca_trimestral = income_plots.plot_quarterly_income_share_lines(
    historial_trimestral_renca
)

numero_trimestres = historial_trimestral_renca[[
    "Ejercicio",
    "Trimestre",
]].drop_duplicates().shape[0]
ancho_grafico_trimestral = max(1200, numero_trimestres * 70 + 240)
fig_renca_trimestral.update_layout(
    width=ancho_grafico_trimestral,
    autosize=False,
)

grafico_trimestral_html = fig_renca_trimestral.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    config={"responsive": False, "displaylogo": False},
)
display(
    HTML(
        f'<div style="max-width: 100%; overflow-x: auto; padding-bottom: 0.5rem;">'
        f"{grafico_trimestral_html}</div>"
    )
)

## Evolución porcentual cuatrimestral desde 2014

In [62]:
reload(income_utils)
reload(income_plots)

historial_cuatrimestral_renca = (
    income_utils.build_municipality_four_month_income_history(
        presupuesto,
        municipality="Renca",
        start_year=2014,
    )
)
fig_renca_cuatrimestral = (
    income_plots.plot_four_month_income_share_lines(
        historial_cuatrimestral_renca
    )
)

numero_cuatrimestres = historial_cuatrimestral_renca[[
    "Ejercicio",
    "Cuatrimestre",
]].drop_duplicates().shape[0]
ancho_grafico_cuatrimestral = max(1200, numero_cuatrimestres * 75 + 240)
fig_renca_cuatrimestral.update_layout(
    width=ancho_grafico_cuatrimestral,
    autosize=False,
)

grafico_cuatrimestral_html = fig_renca_cuatrimestral.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    config={"responsive": False, "displaylogo": False},
)
display(
    HTML(
        f'<div style="max-width: 100%; overflow-x: auto; padding-bottom: 0.5rem;">'
        f"{grafico_cuatrimestral_html}</div>"
    )
)

## Evolución porcentual semestral desde 2014

In [63]:
historial_semestral_renca = (
    income_utils.build_municipality_semiannual_income_history(
        presupuesto,
        municipality="Renca",
        start_year=2014,
    )
)
fig_renca_semestral = income_plots.plot_semiannual_income_share_lines(
    historial_semestral_renca
)

numero_semestres = historial_semestral_renca[[
    "Ejercicio",
    "Semestre",
]].drop_duplicates().shape[0]
ancho_grafico_semestral = max(1200, numero_semestres * 90 + 240)
fig_renca_semestral.update_layout(
    width=ancho_grafico_semestral,
    autosize=False,
)

grafico_semestral_html = fig_renca_semestral.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    config={"responsive": False, "displaylogo": False},
)
display(
    HTML(
        f'<div style="max-width: 100%; overflow-x: auto; padding-bottom: 0.5rem;">'
        f"{grafico_semestral_html}</div>"
    )
)

## Ingresos trimestrales desde 2014

Montos expresados en miles de millones de pesos chilenos.

In [64]:
reload(income_utils)
reload(income_plots)

historial_trimestral_renca = (
    income_utils.build_municipality_quarterly_income_history(
        presupuesto,
        municipality="Renca",
        start_year=2014,
    )
)
fig_dinero_trimestral = (
    income_plots.plot_quarterly_income_amount_lines(
        historial_trimestral_renca
    )
)
numero_trimestres = historial_trimestral_renca[[
    "Ejercicio",
    "Trimestre",
]].drop_duplicates().shape[0]
ancho_grafico_trimestral = max(1200, numero_trimestres * 70 + 240)
fig_dinero_trimestral.update_layout(
    width=ancho_grafico_trimestral,
    autosize=False,
)

grafico_dinero_trimestral_html = fig_dinero_trimestral.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    config={"responsive": False, "displaylogo": False},
)
display(
    HTML(
        f'<div style="max-width: 100%; overflow-x: auto; padding-bottom: 0.5rem;">'
        f"{grafico_dinero_trimestral_html}</div>"
    )
)

## Ingresos cuatrimestrales desde 2014

Montos expresados en miles de millones de pesos chilenos.

In [65]:
historial_cuatrimestral_renca = (
    income_utils.build_municipality_four_month_income_history(
        presupuesto,
        municipality="Renca",
        start_year=2014,
    )
)
fig_dinero_cuatrimestral = (
    income_plots.plot_four_month_income_amount_lines(
        historial_cuatrimestral_renca
    )
)
numero_cuatrimestres = historial_cuatrimestral_renca[[
    "Ejercicio",
    "Cuatrimestre",
]].drop_duplicates().shape[0]
ancho_grafico_cuatrimestral = max(1200, numero_cuatrimestres * 75 + 240)
fig_dinero_cuatrimestral.update_layout(
    width=ancho_grafico_cuatrimestral,
    autosize=False,
)

grafico_dinero_cuatrimestral_html = fig_dinero_cuatrimestral.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    config={"responsive": False, "displaylogo": False},
)
display(
    HTML(
        f'<div style="max-width: 100%; overflow-x: auto; padding-bottom: 0.5rem;">'
        f"{grafico_dinero_cuatrimestral_html}</div>"
    )
)

## Ingresos semestrales desde 2014

Montos expresados en miles de millones de pesos chilenos.

In [66]:
historial_semestral_renca = (
    income_utils.build_municipality_semiannual_income_history(
        presupuesto,
        municipality="Renca",
        start_year=2014,
    )
)
fig_dinero_semestral = (
    income_plots.plot_semiannual_income_amount_lines(
        historial_semestral_renca
    )
)
numero_semestres = historial_semestral_renca[[
    "Ejercicio",
    "Semestre",
]].drop_duplicates().shape[0]
ancho_grafico_semestral = max(1200, numero_semestres * 90 + 240)
fig_dinero_semestral.update_layout(
    width=ancho_grafico_semestral,
    autosize=False,
)

grafico_dinero_semestral_html = fig_dinero_semestral.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    config={"responsive": False, "displaylogo": False},
)
display(
    HTML(
        f'<div style="max-width: 100%; overflow-x: auto; padding-bottom: 0.5rem;">'
        f"{grafico_dinero_semestral_html}</div>"
    )
)

## ¿Qué explica la variación semestral de Otros ingresos?

Se desagrega **Otros ingresos** al nivel de subasignación y se ordenan las cuentas por su variación semestral promedio: $\operatorname{promedio}(|S1-S2|)$ entre 2014 y 2025. Esta medida captura la oscilación entre semestres sin que diferencias positivas y negativas se cancelen. Se muestran las seis cuentas más variables; las demás se agrupan como *Resto de otros ingresos*.

> Los montos son nominales. Una diferencia promedio $S1-S2$ positiva indica que la cuenta suele concentrarse en el primer semestre; una negativa, en el segundo.

In [67]:
reload(income_utils)
reload(income_plots)

desglose_otros_renca, ranking_otros_renca = (
    income_utils.analyze_other_income_semester_variability(
        presupuesto,
        municipality="Renca",
        start_year=2014,
        top_n=6,
    )
)

resumen_variabilidad = (
    ranking_otros_renca.loc[
        ranking_otros_renca["seleccionada"],
        [
            "ranking_variabilidad",
            "codigo_cuenta",
            "cuenta",
            "variacion_promedio_abs_s1_s2",
            "diferencia_promedio_s1_s2",
            "participacion_variabilidad",
        ],
    ]
    .assign(
        variacion_promedio_abs_s1_s2=lambda datos: (
            datos["variacion_promedio_abs_s1_s2"] / 1_000_000_000
        ),
        diferencia_promedio_s1_s2=lambda datos: (
            datos["diferencia_promedio_s1_s2"] / 1_000_000_000
        ),
    )
    .rename(
        columns={
            "ranking_variabilidad": "Ranking",
            "codigo_cuenta": "Código",
            "cuenta": "Cuenta",
            "variacion_promedio_abs_s1_s2": "Promedio |S1-S2| (mil MM)",
            "diferencia_promedio_s1_s2": "Promedio S1-S2 (mil MM)",
            "participacion_variabilidad": "% de variabilidad",
        }
    )
)
resumen_variabilidad.round(
    {
        "Promedio |S1-S2| (mil MM)": 3,
        "Promedio S1-S2 (mil MM)": 3,
        "% de variabilidad": 1,
    }
)

,Ranking,Código,Cuenta,Promedio |S1-S2| (mil MM),Promedio S1-S2 (mil MM),% de variabilidad
0,1,03-2001002,Permisos de Circulación — De Beneficio Fondo C...,1.159,1.159,58.4
1,2,12-10000000,INGRESOS POR PERCIBIR,0.229,0.228,11.5
2,3,08-99999000,Otros,0.188,-0.084,9.5
3,4,08-2006000,Registro de Multas de Tránsito No Pagadas — De...,0.173,0.167,8.7
4,5,08-1002000,Recuperaciones Art. 12 Ley Nº 18.196 y Ley Nº ...,0.099,-0.04,5.0
5,6,08-2002002,"Multas Art. 14, N° 6, Ley N° 18.695 – De Benef...",0.069,0.057,3.5


**Hallazgo con los datos actuales:** el principal impulsor es *Permisos de Circulación — De Beneficio Fondo Común Municipal*. Su diferencia típica entre semestres es de aproximadamente **$1,16 mil millones**, siempre concentrada hacia S1 en este período, y representa **58,4%** de la variabilidad total medida entre subcuentas. Las seis primeras cuentas concentran **96,6%** de esa variabilidad.

> Nota de clasificación: esta es la subasignación `03-2001002`, correspondiente al beneficio del FCM. La parte de beneficio municipal (`03-2001001`) sí está incluida en IPP; por eso esta partida aparece aquí dentro de Otros ingresos y no en la línea IPP.

### Descomposición de cada semestre

Las barras apiladas son la vista principal: permiten ver qué cuentas producen cada subida o bajada. La línea negra conserva la serie total de Otros ingresos como referencia y el gráfico mantiene los ajustes contables negativos bajo la línea cero.

In [68]:
fig_desglose_otros = (
    income_plots.plot_other_income_semester_breakdown(
        desglose_otros_renca
    )
)
numero_semestres_otros = desglose_otros_renca[[
    "Ejercicio",
    "Semestre",
]].drop_duplicates().shape[0]
ancho_desglose_otros = max(1200, numero_semestres_otros * 90 + 360)
fig_desglose_otros.update_layout(
    width=ancho_desglose_otros,
    autosize=False,
)
html_desglose_otros = fig_desglose_otros.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    config={"responsive": False, "displaylogo": False},
)
display(
    HTML(
        f'<div style="max-width: 100%; overflow-x: auto; padding-bottom: 0.5rem;">'
        f"{html_desglose_otros}</div>"
    )
)

### Desglose Leyenda
[desglose renca sinim](https://datos.sinim.gov.cl/impresion_ficha_comunal.php?municipio=13128)

- **Permisos de Circulación — De Beneficio Fondo Común Municipal:** parte de la recaudación de permisos de circulación que la municipalidad debe enterar al Fondo Común Municipal (FCM). Su concentración en S1 es coherente con el calendario anual de renovación de permisos. Aunque Renca recauda el dinero, esta fracción está destinada al FCM y no equivale íntegramente a recursos de libre disponibilidad municipal.

- **Ingresos por percibir:** cobros realizados durante el año por derechos o cuentas que habían sido devengados y permanecían pendientes al cierre de ejercicios anteriores. Pueden variar por el tamaño de la cartera pendiente, la recuperación de deuda y el momento en que se regularizan los pagos.

- **Otros:** subasignación residual oficial dentro de *Otros ingresos corrientes*. Puede reunir entradas que el archivo fuente no desagrega en una cuenta más específica, por lo que su composición exacta no puede inferirse solamente desde esta etiqueta. Los valores negativos pueden corresponder a reversos o ajustes contables y no deberían eliminarse automáticamente.

- **Registro de Multas de Tránsito No Pagadas — De Beneficio Otras Municipalidades:** multas registradas que Renca cobra, por ejemplo durante la regularización asociada a un permiso de circulación, pero cuyo municipio beneficiario es otra comuna. Es principalmente recaudación por cuenta de terceros y puede generar posteriormente una obligación de transferencia.

- **Recuperaciones Art. 12 Ley N.º 18.196 y Ley N.º 19.117, artículo único:** recuperaciones pagadas por FONASA, isapres o cajas de compensación a la municipalidad por subsidios de incapacidad laboral vinculados con licencias médicas de sus funcionarios. La variación depende de las licencias presentadas, su duración y la fecha efectiva del reembolso.

- **Multas Art. 14, N.º 6, Ley N.º 18.695 — De Beneficio Fondo Común Municipal:** recaudación de determinadas multas —especialmente algunas infracciones de tránsito o TAG— cuya normativa destina todo o parte del monto al FCM. Su movimiento refleja el momento del cobro, pero tampoco representa necesariamente un ingreso disponible para gasto propio de Renca.

- **Resto de otros ingresos:** agrupación creada para este gráfico con todas las cuentas que quedaron fuera de las seis más variables. Incluye, entre otras, devoluciones y reintegros, aranceles del Registro de Multas, venta de servicios, reembolsos por licencias médicas, fondos de terceros y otras multas o derechos. Su composición puede cambiar entre años.

- **Total Otros ingresos:** suma contable de todos los componentes anteriores en cada semestre; la línea negra no es una cuenta adicional.

> **Precaución de lectura:** las partidas rotuladas *de beneficio FCM* o *de beneficio otras municipalidades* son montos recaudados por Renca con un destino legal específico. Por eso, el total bruto del gráfico no debe interpretarse automáticamente como caja municipal de libre disposición.

Fuentes normativas de referencia: [Ley N.º 18.196, artículo 12](https://www.bcn.cl/leychile/navegar?idNorma=29617), [Ley N.º 19.117](https://www.bcn.cl/leychile/navegar?i=30484), [Ley N.º 18.695](https://www.bcn.cl/leychile/navegar?idNorma=30077) y [Manual de Procedimientos Contables del Sector Municipal](https://www.sinim.gov.cl/archivos/home/758/Of_E59549_de_2020_Manual_de_Procedimientos_SM.pdf).

## Investigación de variables exógenas (2014–2025)

El gráfico permite proponer mecanismos, pero **no demuestra causalidad por sí solo**. La investigación distingue entre:

- shocks externos a la gestión municipal —leyes, calendario, inflación, actividad económica y situación sanitaria.
- decisiones internas de Renca —captación de permisos, cobranza, canales digitales y ajustes contables. 
   
Esta separación es importante porque varios saltos que parecen macroeconómicos tienen una explicación administrativa documentada.

### Variables externas con mayor sustento

| Variable | Período relevante | Componentes que podría afectar | Mecanismo y lectura esperada | Sustento |
|---|---|---|---|---|
| **Tasación fiscal de vehículos, UTM e inflación** | Todos los años; especialmente 2021–2024 | Permisos de circulación y multas | El permiso se calcula anualmente sobre la tasación fiscal del vehículo y una escala en UTM. Además, todas las series del gráfico están en pesos nominales: una parte del aumento puede ser precio y no crecimiento real. | **Alto**. El [SII explica la base de cálculo](https://www.sii.cl/preguntas_frecuentes/tasac_fiscal_vehiculos/001_170_5079.htm); el [INE permite convertir a pesos reales](https://calculadoraipc.ine.gob.cl/). En 2022 el IPC anual fue 12,8 %, por lo que este control es indispensable. |
| **Cantidad y composición del parque vehicular** | 2014–2025 | Permisos FCM y multas asociadas al permiso | Más vehículos —y vehículos de mayor tasación— amplían la base potencial. Sin embargo, importa más el número de permisos efectivamente emitidos por Renca que el parque nacional. | **Alto como mecanismo**, pendiente una serie comunal completa. El [INE construye la estadística desde registros municipales](https://www.ine.gob.cl/estadisticas-por-tema/comercio-y-servicios/permisos-de-circulacion); a nivel nacional los permisos crecieron 4,6 % en 2023 y 2,9 % en 2024. |
| **Extensión legal del calendario de permisos por COVID-19** | 2020 | Permisos FCM, multas y pagos rezagados | La fecha de renovación 2020 pasó del 31 de marzo al 30 de junio y se alteró el calendario de cuotas. Esto puede desplazar recaudación entre meses y semestres, sin implicar necesariamente un cambio permanente del total anual. | **Alto**. [Ley N.º 21.223](https://www.bcn.cl/leychile/navegar?idNorma=1143999). |
| **Recesión, desempleo y liquidez de los hogares durante la pandemia** | 2020–2021, con rezagos posteriores | Ingresos por percibir, multas y resto de otros ingresos | Menor empleo y movilidad pueden retrasar pagos y aumentar cuentas por cobrar; la reapertura y las ayudas/retiros pueden producir pagos de recuperación. El signo, por tanto, no es necesariamente único. | **Medio**. En 2020 el PIB cayó 6,1 % y la desocupación de la RM llegó a 11,4 % en oct.–dic.; en 2021 el PIB rebotó 11,7 %. Véanse [Banco Central](https://www.bcentral.cl/contenido/-/detalle/informe-de-cuentas-nacionales-de-chile-cuarto-trimestre-de-2021) e [INE](https://www.ine.gob.cl/inicio/2021/01/29/tasa-de-desocupaci%C3%B3n-nacional-alcanz%C3%B3-10-3-en-el-trimestre-octubre-diciembre-de-2020). |
| **Cambios legales para multas TAG y convenios de pago** | 2020–2021 y 2023–2025 | Multas de otras municipalidades y multas de beneficio FCM | Los convenios permiten renovar el permiso pagando una fracción de la deuda y cuotas. Pueden aumentar el número de pagos y cambiar su calendario, aunque reduzcan el monto exigible por infracción. | **Alto como mecanismo**. [Ley N.º 21.213](https://www.bcn.cl/leychile/navegar?idNorma=1142940), su extensión mediante [Ley N.º 21.268](https://www.bcn.cl/leychile/navegar?idNorma=1149645) y el régimen de [Ley N.º 21.547](https://www.bcn.cl/leychile/navegar?idNorma=1190824). |
| **Incidencia y duración de licencias médicas** | 2020–2022 y eventuales olas posteriores | Recuperaciones por licencias médicas | Más días de ausencia cubiertos generan más derechos de reembolso, normalmente con rezago entre aprobación, cobro y registro. | **Medio-alto**. En 2022 las licencias médicas nacionales aumentaron 12,5 % y las asociadas a COVID-19 fueron 17 % del total, según [SUSESO](https://www.suseso.cl/605/w3-article-716398.html). Falta verificar los días aprobados de funcionarios de Renca y la fecha efectiva de cada reembolso. |

**Conclusión principal:** para explicar la tendencia conviene priorizar tasación/UTM, IPC y número de permisos; para explicar quiebres, usar indicadores legales de 2020 y 2023 y licencias médicas. No obstante, las cuentas genéricas `Otros` e `Ingresos por percibir` no deben asociarse automáticamente a una variable macroeconómica: pueden contener reconocimientos y regularizaciones contables puntuales.

### Contraste con los semestres anómalos de Renca

Los siguientes montos provienen de la serie mensual utilizada en este notebook y se expresan en **millones de pesos nominales**.

| Hallazgo en la serie | Explicación más defendible | Clasificación |
|---|---|---|
| **2018 S1 — Ingresos por percibir: $1.182 MM**, de los cuales $1.079 MM aparecen en junio | El grado de concentración apunta a recuperación o reconocimiento de una cuenta específica, no a una variación macroeconómica gradual. Se necesita el mayor contable y el comprobante de junio. | **No identificado; probablemente contable/administrativo** |
| **2018 S2 — Otros: –$158 MM**, con –$198 MM en diciembre | Un ingreso negativo es señal de reverso, reclasificación o corrección. No debe modelarse como respuesta económica ordinaria. | **Ajuste contable probable** |
| **2020 — Permisos FCM mantienen un monto alto en S2 ($1.537 MM)** | Es compatible con el calendario excepcional de la Ley 21.223. Sin embargo, Renca informa además que su recaudación total de permisos aumentó 30 % porque mantuvo módulos presenciales e implementó el perdonazo TAG, atrayendo contribuyentes de otras comunas. | **Shock legal externo + respuesta interna** |
| **2022 S2 — Recuperaciones médicas: $502 MM** | Es consistente con el aumento nacional de licencias y con cobros rezagados, pero la coincidencia temporal no prueba causalidad. Debe verificarse con días de licencia aprobados, organismo pagador y fecha de recepción. | **Hipótesis externa plausible** |
| **2023 S2 — Otros: $1.028 MM**, de los cuales $874 MM aparecen en julio | La Cuenta Pública 2023 informa que $962,9 MM de fondos sin origen identificado, depositados antes de 2018 y no reclamados, se incorporaron al patrimonio municipal. La magnitud explica plausiblemente gran parte del salto, sujeto a confirmar la imputación exacta. | **Regularización contable documentada; no exógena** |
| **2023–2025 — aumento y concentración en S1 de las dos series de multas** | La renovación de permisos concentra pagos en marzo y la Ley 21.547 habilitó convenios TAG. Renca reportó 1.446 convenios en 2023 y $195 MM adicionales en multas, pero también una estrategia de cobranza iniciada en 2020. | **Ley externa + cobranza interna** |
| **2023 — fuerte aumento de permisos** | Renca pasó de 65.754 giros y $7.278 MM en 2022 a 84.802 giros y $10.673 MM en 2023. La propia municipalidad lo vincula a captación de empresas, traslado de patentes, atención digital y nuevos puntos de venta. | **Principalmente gestión interna, no shock exógeno** |

Fuentes locales: [Cuenta Pública de Renca 2020](https://renca.cl/wp-content/uploads/2021/04/Cuenta-Pu%CC%81blica-2020-Municipalidad-de-Renca.pdf) y [Cuenta Pública de Renca 2023](https://renca.cl/wp-content/uploads/2025/05/Cuenta-publica-2023.pdf). Estas fuentes permiten evitar una conclusión engañosa: **la inflación y las leyes importan, pero los mayores quiebres de 2020 y 2023 también reflejan decisiones de gestión y regularizaciones contables de Renca**.

### Cómo llevar estas hipótesis a un análisis cuantitativo

Antes de estimar efectos o construir un forecast, convendría incorporar, en este orden:

1. **Deflactor mensual:** IPC o UTM, para separar crecimiento nominal y real.
2. **Base vehicular de Renca:** número de giros por tipo de vehículo y, si es posible, tasación fiscal promedio/mediana.
3. **Calendario e hitos legales:** marzo, segunda cuota y variables indicadoras para Ley 21.223 (2020) y los regímenes TAG de 2020 y 2023.
4. **Stock y flujo de multas:** multas ingresadas al registro, convenios firmados, cuotas pagadas y comuna beneficiaria.
5. **Licencias médicas municipales:** días aprobados, monto solicitado, monto recuperado y rezago por COMPIN/ISAPRE.
6. **Ciclo económico de la RM:** empleo/desempleo y actividad, idealmente con uno a tres meses de rezago.
7. **Controles internos:** apertura de canales digitales, campañas de captación, intensidad de cobranza y marcadores de ajustes contables.

Hay sólo **24 observaciones semestrales** entre 2014 y 2025, insuficientes para estimar muchos efectos simultáneos. La unidad recomendable es mensual —144 observaciones—, con pocos predictores, rezagos definidos por el mecanismo y validación temporal. Los montos negativos o extraordinarios deben etiquetarse como ajustes/intervenciones; no conviene borrarlos ni permitir que el modelo los interprete como estacionalidad normal.

### Ranking de cuentas más variables

Este ranking complementa las barras: compara directamente la magnitud típica de la diferencia entre S1 y S2, independientemente del signo.

In [69]:
fig_ranking_otros = (
    income_plots.plot_other_income_semester_variability(
        ranking_otros_renca,
        top_n=6,
    )
)
fig_ranking_otros.show(config={"displaylogo": False})